In [ ]:
import time
import nb_setup
from src.data import load_tokenised_data
from deep_translator import GoogleTranslator

In [ ]:
df_train_tokenised, df_val_tokenised = load_tokenised_data()

In [ ]:
# 1. Number of examples, answerable/unanswerable proportions, median and interquantile range of
# tokenised question and context lengths (table sufficient, plots are optional), missing values, and
# exact duplicate question--context pairs. Everything by language and dataset split (train/validation).


def statistics_by_language(df, dataset_name):
    stats = df.groupby("lang").agg(
        num_examples=("lang", "size"),
        num_answerable=("answerable", "sum"),
        num_unanswerable=("answerable", lambda x: (~x).sum()),
        median_question_length=("question_tokens", lambda x: x.apply(len).median()),
        iqr_question_length=("question_tokens", lambda x: x.apply(len).quantile(0.75) - x.apply(len).quantile(0.25)),
        median_context_length=("context_tokens", lambda x: x.apply(len).median()),
        iqr_context_length=("context_tokens", lambda x: x.apply(len).quantile(0.75) - x.apply(len).quantile(0.25)),
        num_missing_values=("question", lambda x: x.isnull().sum() + df["context"].isnull().sum() + df["answer"].isnull().sum()),
        num_exact_duplicates=("question", lambda x: df.duplicated(subset=["question", "context"]).sum())
    ).reset_index()
    stats["answerable_proportion"] = (
        stats["num_answerable"] / stats["num_examples"]
    ).round(4)

    stats["unanswerable_proportion"] = (
        stats["num_unanswerable"] / stats["num_examples"]
    ).round(4)
    stats["dataset"] = dataset_name
    return stats


In [ ]:
stats_train = statistics_by_language(df_train_tokenised, "train")
stats_val = statistics_by_language(df_val_tokenised, "validation")

In [ ]:
stats_train

In [ ]:
stats_val

In [ ]:
# 2. Five most common question tokens and their counts for each language, together with English translation,
# and explain your tokenisation.

five_most_common_question_tokens = {}
for lang in df_train_tokenised["lang"].unique():
    lang_df = df_train_tokenised[df_train_tokenised["lang"] == lang]
    question_tokens = lang_df["question_tokens"].explode()
    most_common_tokens = question_tokens.value_counts().head(5)
    five_most_common_question_tokens[lang] = most_common_tokens
    for token, count in most_common_tokens.items():
        translated_token = GoogleTranslator(source='auto', target='en').translate(token)
        print(f"Token: {token}, Count: {count}, Translated: {translated_token}")
        five_most_common_question_tokens[lang][token] = (count, translated_token)
    time.sleep(2)
